# Limpieza de las mediciones de calidad del aire (2024 y 2025)

Este cuaderno produce `Data/datos_24_25.csv`, el archivo de mediciones que leen las dos vistas de la app.

**Entradas.** Dos exportaciones de BigQuery con las mediciones horarias de las estaciones (≈ 1.3 GB cada una):

| Archivo | Periodo |
| :--- | :--- |
| `bq-results-20251016-165053-1760633522998.csv` | 1 ene – 21 may 2024 |
| `bq-results-20251016-165608-1760633792315.csv` | 23 ene – 27 may 2025 |

Ambos tienen las columnas `date_time, variable, value, unit_measurement, variable_type, station`, en formato largo (una fila por estación, hora y variable).

**Pasos.**

1. Filtrar las 10 variables que usa la visualización.
2. Quitar los registros duplicados (filas idénticas en todas las columnas).
3. Quitar valores inválidos: códigos de dato faltante (9999), saturación del equipo (985), valores fuera de rango físico y series de sensores dañados.
4. Resolver las pocas horas que quedan con dos valores distintos para la misma estación y variable.
5. Asignar la unidad de medida de cada variable.
6. Validar y guardar.

**Por qué el resultado pesa ≈ 17 MB.** Las exportaciones traen cada medición repetida unas 95 veces, y además incluyen variables que no usamos. En la sección 2 se muestra la cuenta.

El cuaderno lee los CSV por bloques, así que corre en un portátil sin Spark. Tarda unos minutos.

## 0. Configuración

In [1]:
import re
import sys
from collections import Counter
from pathlib import Path

import pandas as pd

pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 140)

# --- Rutas -------------------------------------------------------------------------
# Localmente se asume la estructura del proyecto:
#   Proyecto Final/
#   ├── CalidadAireIndustrias_2025/raw_data/   <- CSV crudos de BigQuery
#   └── codigo/Web-App-Calidad-de-Aire-e-Industrias/
#       ├── Preprocessing/Limpiar_datos.ipynb  <- este cuaderno
#       └── Data/datos_24_25.csv               <- salida
# En Colab se monta Drive y hay que ajustar las dos rutas de abajo.
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    RAW_DIR = Path("/content/drive/MyDrive/<carpeta del proyecto>/raw_data")
    SALIDA = Path("/content/drive/MyDrive/<carpeta del proyecto>/datos_24_25.csv")
else:
    RAW_DIR = Path("../../../CalidadAireIndustrias_2025/raw_data")
    SALIDA = Path("../Data/datos_24_25.csv")

ARCHIVOS = {
    "2024": RAW_DIR / "bq-results-20251016-165053-1760633522998.csv",
    "2025": RAW_DIR / "bq-results-20251016-165608-1760633792315.csv",
}
for anio, ruta in ARCHIVOS.items():
    assert ruta.exists(), f"No se encontró el archivo de {anio}: {ruta.resolve()}"

# Filas por bloque al leer. Con 2 millones el proceso usa menos de 2 GB de RAM.
FILAS_POR_BLOQUE = 2_000_000

COLUMNAS = ["date_time", "variable", "value", "unit_measurement", "variable_type", "station"]

### Variables y unidades

Los archivos crudos traen `unit_measurement = unknown` en todas las filas, así que la unidad se asigna a mano.
Se asume que los contaminantes vienen en µg/m³, que es lo que reportan las redes de monitoreo y lo que espera la app para calcular el ICA.

In [2]:
# código en el CSV -> (nombre legible, unidad)
VARIABLES = {
    "PM2":  ("PM2.5", "µg/m³"),
    "PM10": ("PM10", "µg/m³"),
    "O3":   ("O₃", "µg/m³"),
    "No2":  ("NO₂", "µg/m³"),
    "So2":  ("SO₂", "µg/m³"),
    "Co":   ("CO", "µg/m³"),
    "Temp": ("Temperatura", "°C"),
    "WDS":  ("Velocidad del viento", "m/s"),
    "WDD":  ("Dirección del viento", "°"),   # grados desde el norte, en sentido horario
    "RAIN": ("Precipitación", "mm"),         # acumulada en la hora
}

### Reglas para valores inválidos

* **Códigos de dato faltante.** El sistema de origen escribe `9999` (y en PM, `9999999`) cuando no hay medición.
* **Saturación del equipo.** Los monitores de PM2.5 y PM10 escriben exactamente `985` cuando llegan a su tope. Hay miles de lecturas idénticas de 985, lo que no ocurre con una medición real.
* **Rango físico.** Para los contaminantes, el tope es el límite superior de la escala ICA (Resolución 2254 de 2017, ICA = 500): un valor por encima no tiene categoría y se trata como error del equipo. Los gases se pasan de ppm a µg/m³ a 25 °C y 1 atm, igual que en `src/pages/expert/domain/icaLevels.ts`. Para las variables meteorológicas se usan límites físicos para Cundinamarca.

No se usa una regla genérica de «≥ 9000 es faltante»: con CO borraría picos reales de 9 a 35 mg/m³ (9 000 – 35 000 µg/m³) que ocurren en horas pico en estaciones urbanas.

In [3]:
CODIGOS_FALTANTE = {9999.0, 9999999.0}
SATURACION_PM = 985.0
VARIABLES_PM = {"PM2", "PM10"}


def ppm_a_ugm3(ppm: float, masa_molar: float) -> float:
    """Convierte ppm a µg/m³ a 25 °C y 1 atm (volumen molar 24.45 L)."""
    return ppm * masa_molar * 1000 / 24.45


# (mínimo, máximo) aceptados, en la unidad de VARIABLES.
LIMITES = {
    "PM2":  (0, 500.4),                       # tope ICA PM2.5 (24 h)
    "PM10": (0, 604),                         # tope ICA PM10 (24 h)
    "O3":   (0, ppm_a_ugm3(0.604, 48.00)),    # tope ICA O₃ 1 h  ≈ 1 186
    "No2":  (0, ppm_a_ugm3(2.04, 46.0055)),   # tope ICA NO₂ 1 h ≈ 3 839
    "So2":  (0, ppm_a_ugm3(1.004, 64.066)),   # tope ICA SO₂     ≈ 2 631
    "Co":   (0, ppm_a_ugm3(50.4, 28.01)),     # tope ICA CO 8 h  ≈ 57 741
    "Temp": (-10, 45),
    "WDS":  (0, 60),
    "WDD":  (0, 360),
    "RAIN": (0, 100),                         # mm en una hora
}
pd.DataFrame(LIMITES, index=["mínimo", "máximo"]).T.round(1)

,mínimo,máximo
PM2,0.0,500.4
PM10,0.0,604.0
O3,0.0,1185.8
No2,0.0,3838.5
So2,0.0,2630.8
Co,0.0,57738.4
Temp,-10.0,45.0
WDS,0.0,60.0
WDD,0.0,360.0
RAIN,0.0,100.0


## 1. Lectura y filtro de variables

Se lee cada archivo por bloques. En cada bloque se cuentan todas las variables (para ver qué se descarta), se filtran las 10 que usamos y se cuenta cuántas veces aparece cada fila. Contar en lugar de guardar todas las copias es lo que permite procesar los 2.6 GB con poca memoria: el conteo por bloque ya es pequeño y al final se suman los conteos de todos los bloques.

In [4]:
def leer_archivo(ruta: Path):
    """Devuelve (filas totales, conteo por variable, copias por fila única de las 10 variables)."""
    filas = 0
    por_variable = Counter()
    copias = []
    for bloque in pd.read_csv(ruta, dtype=str, chunksize=FILAS_POR_BLOQUE):
        filas += len(bloque)
        por_variable.update(bloque["variable"].value_counts(dropna=False).to_dict())
        usadas = bloque[bloque["variable"].isin(VARIABLES)]
        # value_counts sobre todas las columnas = cuántas veces aparece cada fila idéntica.
        copias.append(usadas[COLUMNAS].value_counts(dropna=False))
    copias = pd.concat(copias).groupby(level=list(range(len(COLUMNAS))), dropna=False).sum()
    return filas, por_variable, copias


crudos = {}
for anio, ruta in ARCHIVOS.items():
    crudos[anio] = leer_archivo(ruta)
    print(f"{anio}: {crudos[anio][0]:,} filas leídas de {ruta.name}")

2024: 25,755,988 filas leídas de bq-results-20251016-165053-1760633522998.csv
2025: 22,406,288 filas leídas de bq-results-20251016-165608-1760633792315.csv


In [5]:
# Todas las variables de los archivos crudos; las que no usamos se descartan.
conteo_variables = pd.DataFrame({anio: datos[1] for anio, datos in crudos.items()}).fillna(0).astype(int)
conteo_variables["se usa"] = conteo_variables.index.isin(VARIABLES)
conteo_variables.sort_values(["se usa", "2024"], ascending=False)

,2024,2025,se usa
So2,2951558,2673430,True
WDS,2605121,2010681,True
WDD,2204206,1826529,True
PM2,1958371,1545932,True
No2,1921145,2784607,True
Co,1854110,1192384,True
Temp,1756651,760286,True
PM10,1562648,1214986,True
O3,758080,367854,True
RAIN,753579,585570,True


## 2. Duplicados

Se consideran duplicadas las filas idénticas en **todas** las columnas. La tabla muestra cuántas copias trae cada medición: la exportación de BigQuery repite cada fila decenas de veces (probablemente por un *join* en la consulta original), y esa es la razón principal de que 2.6 GB se reduzcan a unos pocos MB sin perder información.

In [6]:
resumen_dup = []
partes = []
for anio, (filas, por_variable, copias) in crudos.items():
    filtradas = int(copias.sum())
    resumen_dup.append({
        "año": anio,
        "filas crudas": filas,
        "filas de las 10 variables": filtradas,
        "filas únicas": len(copias),
        "duplicados quitados": filtradas - len(copias),
        "copias por fila (mediana)": int(copias.median()),
        "copias por fila (mín – máx)": f"{copias.min()} – {copias.max()}",
    })
    partes.append(copias.index.to_frame(index=False))

display(pd.DataFrame(resumen_dup).set_index("año").T)

datos = pd.concat(partes, ignore_index=True)
# Un mismo registro no aparece en los dos archivos, pero se verifica igual.
antes = len(datos)
datos = datos.drop_duplicates(ignore_index=True)
print(f"Duplicados entre archivos: {antes - len(datos):,}")
print(f"Filas tras quitar duplicados: {len(datos):,}")

año,2024,2025
filas crudas,25755988,22406288
filas de las 10 variables,18325469,14962259
filas únicas,178289,165280
duplicados quitados,18147180,14796979
copias por fila (mediana),80,80
copias por fila (mín – máx),1 – 9030,2 – 238


Duplicados entre archivos: 0
Filas tras quitar duplicados: 343,569


In [7]:
# Columnas sin información: deben tener un solo valor en todo el archivo.
for columna in ["unit_measurement", "variable_type"]:
    print(columna, "->", datos[columna].unique().tolist())
assert datos[["date_time", "variable", "value", "station"]].notna().all().all(), "Hay celdas vacías"

datos["valor"] = pd.to_numeric(datos["value"], errors="raise")
datos["anio"] = datos["date_time"].str[:4]

unit_measurement -> ['unknown']
variable_type -> ['hourly']


## 3. Valores inválidos

### 3.1 Series de sensores dañados

Antes de mirar valores sueltos se revisa cada serie (estación × variable × año). Aparecen dos casos:

* Series que son **9999 de principio a fin**: la estación no tiene ese sensor (estación 15 para PM2.5 y estación 21 para SO₂). Explican casi todos los 9999 de los datos.
* Series con un comportamiento imposible: un valor constante durante semanas, o una distribución que no corresponde a la variable.

In [8]:
def describir(g: pd.Series) -> pd.Series:
    v = g[~g.isin(CODIGOS_FALTANTE)]
    return pd.Series({
        "horas": len(g),
        "valores distintos": v.nunique(),
        "% ceros": round(100 * (v == 0).mean(), 1),
        "mediana": round(v.median(), 2),
        "p99": round(v.quantile(0.99), 2),
        "máximo": round(v.max(), 2),
    })


series = datos.groupby(["station", "variable", "anio"])["valor"].apply(describir).unstack()
variable = series.index.get_level_values("variable")

# Series que son 9999 de principio a fin: la estación no tiene ese sensor.
# La regla del código 9999 (3.2) las elimina; se listan solo como referencia.
sin_sensor = series[series["valores distintos"] == 0]
print("Series completas en 9999 (estación sin ese sensor):")
display(sin_sensor[["horas"]])

sospechosas = series[
    # Un único valor durante más de una semana. La lluvia se excluye: puede ser 0 por semanas.
    ((series["valores distintos"] == 1) & (series["horas"] > 168) & (variable != "RAIN"))
    # Lluvia con mediana > 1 mm/h: en una serie real la mayoría de horas no llueve.
    | ((variable == "RAIN") & (series["mediana"] > 1))
    # Temperatura que nunca pasa de 1 °C: imposible en las estaciones de Cundinamarca.
    | ((variable == "Temp") & (series["p99"] < 1))
]
print("Series sospechosas:")
sospechosas

Series completas en 9999 (estación sin ese sensor):


horas
station variable anio        
11      So2      2024     1.0
15      PM2      2024  1439.0
                 2025  2362.0
21      So2      2024  2407.0

Series sospechosas:


horas  valores distintos  % ceros  mediana      p99   máximo
station variable anio                                                               
15      RAIN     2024  1439.0              870.0     37.2    19.13  1055.80  1148.56
                 2025  2361.0             1513.0     32.1    24.80  1053.38  1199.76
17      Temp     2024  1441.0             1357.0      0.0     0.02     0.02     0.03
4       WDD      2025  2604.0                1.0    100.0     0.00     0.00     0.00

Las tres series se descartan completas:

* **Estación 15, precipitación:** mediana de 24 mm y p90 de 739 mm *por hora*. No es precipitación; probablemente es otro sensor etiquetado como `RAIN`.
* **Estación 17, temperatura:** todos los valores están entre 0.01 y 0.02 °C.
* **Estación 4, dirección del viento en 2025:** vale 0 durante todo el periodo.

Hay otros patrones que se dejan pero conviene tener en cuenta al interpretar: la estación 18 tiene ~1 800 horas con velocidad del viento exactamente 0.3 m/s (umbral mínimo del anemómetro, es decir, calma) y la estación 7 tiene ~1 000 horas con NO₂ = 0 (por debajo del límite de detección).

In [9]:
# (estación, variable, año o None para todos los años) -> motivo
SERIES_DANADAS = {
    ("15", "RAIN", None): "precipitación imposible (mediana 24 mm/h)",
    ("17", "Temp", None): "temperatura fija entre 0.01 y 0.02 °C",
    ("4", "WDD", "2025"): "dirección del viento siempre 0",
}

# Si la exportación cambia, esta verificación avisa que la lista hay que revisarla.
detectadas = {(s, v) for s, v, _ in sospechosas.index}
listadas = {(s, v) for s, v, _ in SERIES_DANADAS}
assert detectadas == listadas, f"Revisar SERIES_DANADAS: detectadas {detectadas}, listadas {listadas}"

### 3.2 Clasificación de cada valor inválido

Cada fila recibe el primer motivo que aplique. Las filas con motivo se eliminan: en formato largo, una hora sin fila es una hora sin dato, que es como la app trata los faltantes.

In [10]:
def motivo_invalido(datos: pd.DataFrame) -> pd.Series:
    motivo = pd.Series(pd.NA, index=datos.index, dtype="string")

    def marcar(condicion: pd.Series, texto: str) -> None:
        motivo[condicion & motivo.isna()] = texto

    for (estacion, variable, anio), texto in SERIES_DANADAS.items():
        cond = (datos["station"] == estacion) & (datos["variable"] == variable)
        if anio is not None:
            cond &= datos["anio"] == anio
        marcar(cond, "serie de sensor dañado")

    marcar(datos["valor"].isin(CODIGOS_FALTANTE), "código de dato faltante (9999)")
    marcar(datos["variable"].isin(VARIABLES_PM) & (datos["valor"] == SATURACION_PM), "saturación del equipo (985)")

    minimo = datos["variable"].map(lambda v: LIMITES[v][0])
    maximo = datos["variable"].map(lambda v: LIMITES[v][1])
    marcar((datos["valor"] < minimo) | (datos["valor"] > maximo), "fuera de rango físico")
    return motivo


datos["motivo"] = motivo_invalido(datos)

invalidos = datos[datos["motivo"].notna()]
tabla = invalidos.pivot_table(index="variable", columns="motivo", values="valor", aggfunc="size", fill_value=0)
tabla["total"] = tabla.sum(axis=1)
tabla.loc["total"] = tabla.sum()
tabla

motivo,código de dato faltante (9999),fuera de rango físico,saturación del equipo (985),serie de sensor dañado,total
variable,,,,,
Co,0,1,0,0,1
No2,4,0,0,0,4
O3,0,4,0,0,4
PM10,19,26,2266,0,2311
PM2,3813,90,912,0,4815
RAIN,0,4,0,3800,3804
So2,2411,1,0,0,2412
Temp,0,0,0,1441,1441
WDD,0,0,0,2604,2604


In [11]:
# Ejemplos de valores fuera de rango, para comprobar que son errores y no picos reales.
(invalidos[invalidos["motivo"] == "fuera de rango físico"]
 .sort_values("valor", ascending=False)
 .groupby("variable").head(3)[["date_time", "station", "variable", "valor"]])

,date_time,station,variable,valor
184619,2025-01-28 09:00:00 UTC,2,PM10,5.787794e+06
158433,2024-05-02 01:00:00 UTC,2,PM10,4.416068e+06
125706,2024-04-03 19:00:00 UTC,22,PM2,4.191568e+06
184674,2025-01-28 10:00:00 UTC,2,PM10,3.553012e+06
132339,2024-04-09 06:00:00 UTC,22,PM2,2.370313e+06
125354,2024-04-03 12:00:00 UTC,22,PM2,1.420539e+06
143623,2024-04-17 11:00:00 UTC,7,Co,1.374603e+05
33484,2024-01-25 21:00:00 UTC,7,O3,1.349538e+04
219946,2025-03-04 10:00:00 UTC,7,O3,3.444816e+03
231549,2025-03-12 19:00:00 UTC,7,O3,3.312534e+03


In [12]:
validos = datos[datos["motivo"].isna()].drop(columns="motivo")
print(f"Filas inválidas quitadas: {len(invalidos):,} ({len(invalidos) / len(datos):.1%})")
print(f"Filas válidas: {len(validos):,}")

Filas inválidas quitadas: 17,396 (5.1%)
Filas válidas: 326,173


## 4. Horas con dos valores distintos

Después de quitar duplicados exactos todavía puede haber dos filas para la misma estación, variable y hora con valores distintos (por ejemplo `1.53904` y `1.539039`, o `0` y `70.86`). Se resuelven así:

1. Si una de las lecturas es 0 y otra no, el 0 se toma como caída del sensor y se descarta.
2. Si quedan varias, se promedian.

In [13]:
CLAVE = ["date_time", "variable", "station"]
repetidas = validos[validos.duplicated(CLAVE, keep=False)].sort_values(CLAVE)
print(f"Horas con más de un valor: {repetidas[CLAVE].drop_duplicates().shape[0]}")
display(repetidas[CLAVE + ["valor"]])

# Regla 1: quitar los ceros de una hora que también tiene una lectura distinta de 0.
hay_no_cero = validos["valor"].ne(0).groupby([validos[c] for c in CLAVE]).transform("any")
validos = validos[~(validos.duplicated(CLAVE, keep=False) & (validos["valor"] == 0) & hay_no_cero)]
# Regla 2: promediar lo que quede repetido.
limpio = validos.groupby(CLAVE, as_index=False).agg(valor=("valor", "mean"), anio=("anio", "first"))
print(f"Filas tras resolverlas: {len(limpio):,}")

Horas con más de un valor: 3


,date_time,variable,station,valor
34110,2024-01-26 06:00:00 UTC,So2,4,1.539039
34111,2024-01-26 06:00:00 UTC,So2,4,1.539040
133905,2024-04-10 11:00:00 UTC,So2,15,0.000000
133912,2024-04-10 11:00:00 UTC,So2,15,70.863670
153699,2024-04-23 14:00:00 UTC,So2,15,0.000000
153702,2024-04-23 14:00:00 UTC,So2,15,17.914993


Filas tras resolverlas: 326,170


## 5. Unidades de medida

In [14]:
limpio["unit_measurement"] = limpio["variable"].map(lambda v: VARIABLES[v][1])
limpio["variable_type"] = "hourly"
limpio.groupby("variable")["unit_measurement"].first().to_frame()

,unit_measurement
variable,
Co,µg/m³
No2,µg/m³
O3,µg/m³
PM10,µg/m³
PM2,µg/m³
RAIN,mm
So2,µg/m³
Temp,°C
WDD,°


## 6. Validación y resumen

In [15]:
assert set(limpio["variable"]) <= set(VARIABLES)
assert not limpio.duplicated(CLAVE).any(), "Quedaron horas repetidas"
assert limpio["date_time"].str.fullmatch(r"\d{4}-\d{2}-\d{2} \d{2}:00:00 UTC").all(), "Formato de fecha inesperado"
assert limpio["station"].str.fullmatch(r"\d+").all()
for variable, (minimo, maximo) in LIMITES.items():
    v = limpio.loc[limpio["variable"] == variable, "valor"]
    assert v.between(minimo, maximo).all(), f"{variable} fuera de rango"
print("Validaciones correctas")

# Resumen por variable del archivo final.
resumen = limpio.groupby("variable")["valor"].describe(percentiles=[0.5, 0.99])
resumen.insert(0, "nombre", [VARIABLES[v][0] for v in resumen.index])
resumen.insert(1, "unidad", [VARIABLES[v][1] for v in resumen.index])
resumen.round(2)

Validaciones correctas


,nombre,unidad,count,mean,std,min,50%,99%,max
variable,,,,,,,,,
Co,CO,µg/m³,18264.0,699.27,808.09,0.36,564.24,2387.62,35207.18
No2,NO₂,µg/m³,27253.0,35.21,26.40,0.00,29.37,107.16,623.18
O3,O₃,µg/m³,5699.0,20.98,34.12,0.02,10.01,179.08,604.38
PM10,PM10,µg/m³,47025.0,35.21,33.96,0.00,27.25,171.00,563.64
PM2,PM2.5,µg/m³,46454.0,19.62,21.31,0.00,14.79,100.44,499.88
RAIN,Precipitación,mm,40300.0,0.12,1.52,0.00,0.00,2.31,84.00
So2,SO₂,µg/m³,38302.0,15.87,51.00,0.00,4.69,268.93,1112.70
Temp,Temperatura,°C,17547.0,15.84,3.29,5.25,15.35,26.38,31.17
WDD,Dirección del viento,°,42663.0,166.64,91.10,0.05,157.69,354.14,360.00


In [16]:
# Horas con dato por estación y variable: muestra la cobertura real de cada estación.
cobertura = limpio.pivot_table(index=["anio", "station"], columns="variable", values="valor", aggfunc="size", fill_value=0)
cobertura = cobertura.sort_index(level=[0, 1], key=lambda i: i.astype(int))
cobertura

variable        Co   No2    O3  PM10   PM2  RAIN   So2  Temp   WDD   WDS
anio station                                                            
2024 2           0    37  1801  2350  2358  2362  2219     0  2362  2362
     3        2643  2622     0  2690  2692  2724  2291  2724  2724  2724
     4        1469     0     0  1564  1565  1565  1285  1565  1565  1565
     5        2250     0     0   831     0  2282  2085     0  2283  2283
     7        1533  1733  2036  1591  2037  2037   910     0  2037  2037
     8           0  2521     0     0  2527     0     0     0     0     0
     9           0     0     0  2266  2266     0     1     0     0     0
     10          0     0     0     0   297     0   329     0     0     0
     11          0     0     0  1287  1287  1287     0  1287  1287  1287
     12          0    97     0    97    83    97    97     0    97    97
     14          0     5     0  2639  2638  2639  2502  2639  2639  2639
     15          0   140     0  1438     0     0  1438     0  1439  1439
     17          0  1441     0  1441     0  1441  1441     0     0     0
     18          0     0     0  2135  1459  2471     0     0  2471  2471
     20          0     0     0  1275  1275  1275  1225     0  1275  1275
     21          0     0     0   469   469     0     0     0     0     0
     22       2401  1903     0     0  2306  2401  1988  2401  2401  2401
     23          0     0     0   473   473   473     0   473   473   473
2025 2           0  2668   761  2645  2669  2669  2628     0  2669  2669
     3        2603  2201     0  2604  2604  2604  2275  2604  2604  2604
     4        2282     0     0  2604  2604     0  2298     0     0     0
     5         563     0     0     0     0   563   107     0   565   565
     7        2520  2516  1101  2509  2090  2520  2518     0  2520  2520
     8           0  2653     0     0  2655     0   603     0     0     0
     9           0     0     0  2676  2676     0  2366     0     0     0
     10          0  1951     0  1958  1955     0  1983     0     0     0
     14          0  2405     0  2008     0  2405  2387  2405  2405  2405
     15          0  2360     0  2330     0     0   959     0  2361  2361
     18          0     0     0  2031  2356  2559     0     0  2560  2560
     20          0     0     0  2477  2477  2477  2367     0  2477  2477
     23          0     0     0   637   636  1449     0  1449  1449  1449

In [17]:
etapas = pd.DataFrame(
    [
        ("Filas crudas", sum(d[0] for d in crudos.values())),
        ("Filas de las 10 variables", sum(int(d[2].sum()) for d in crudos.values())),
        ("Sin duplicados", len(datos)),
        ("Sin valores inválidos", len(datos) - len(invalidos)),
        ("Una fila por estación, variable y hora", len(limpio)),
    ],
    columns=["etapa", "filas"],
).set_index("etapa")
etapas["% de la etapa anterior"] = (100 * etapas["filas"] / etapas["filas"].shift()).round(2)
etapas

,filas,% de la etapa anterior
etapa,,
Filas crudas,48162276,NaN
Filas de las 10 variables,33287728,69.12
Sin duplicados,343569,1.03
Sin valores inválidos,326173,94.94
"Una fila por estación, variable y hora",326170,100.00


## 7. Guardar

Se conserva el formato que lee la app (`src/data/observations.ts` y `src/pages/expert/data/measurements.ts` leen las columnas por posición): mismas seis columnas, en el mismo orden, con la fecha como texto `AAAA-MM-DD HH:00:00 UTC`. Las filas se ordenan por fecha, estación y variable para que el archivo sea reproducible.

In [18]:
salida = (
    limpio.assign(value=limpio["valor"].round(6), orden=limpio["station"].astype(int))
    .sort_values(["date_time", "orden", "variable"])[COLUMNAS]
)
SALIDA.parent.mkdir(parents=True, exist_ok=True)
# lineterminator fijo: el archivo queda igual sin importar el sistema operativo.
salida.to_csv(SALIDA, index=False, encoding="utf-8", lineterminator="\n")
print(f"Guardado {SALIDA}: {len(salida):,} filas, {SALIDA.stat().st_size / 1e6:.1f} MB")
salida.head()

Guardado ..\Data\datos_24_25.csv: 326,170 filas, 17.0 MB


,date_time,variable,value,unit_measurement,variable_type,station
8,2024-01-01 00:00:00 UTC,O3,1.449369,µg/m³,hourly,2
14,2024-01-01 00:00:00 UTC,PM10,31.358500,µg/m³,hourly,2
23,2024-01-01 00:00:00 UTC,PM2,23.123130,µg/m³,hourly,2
32,2024-01-01 00:00:00 UTC,RAIN,0.021528,mm,hourly,2
41,2024-01-01 00:00:00 UTC,So2,1.751358,µg/m³,hourly,2
